In [ ]:
import pandas as pd
import numpy as np
import re

# === 1) READ CSV ===
file_path = "home_sdf_marketing_sample_for_amazon_in-ecommerce__20191001_20191031__30k_data.csv"
df = pd.read_csv(file_path)

# === 2) SELECT REQUIRED COLUMNS & RENAME ===
work = df[[
    "Product Title",
    "Product Description",
    "Category",
    "Mrp",
    "Price",
    "Image Urls"
]].copy()

work = work.rename(columns={
    "Product Title": "title",
    "Product Description": "description",
    "Category": "category",
    "Mrp": "msrp_raw",
    "Price": "price_raw",
    "Image Urls": "image_urls_raw"
})

# === 3) CLEAN PRICE FIELDS ===
def to_float_safe(x):
    if pd.isna(x):
        return np.nan
    s = re.sub(r"[^0-9.]", "", str(x))
    return float(s) if s else np.nan

work["price"] = work["price_raw"].apply(to_float_safe)
work["msrp"] = work["msrp_raw"].apply(to_float_safe)
work = work.dropna(subset=["title", "category", "price"])

# === 4) EXTRACT FIRST VALID IMAGE URL ===
def extract_image(cell):
    if pd.isna(cell): return np.nan
    urls = re.findall(r"https?://\\S+", str(cell))
    return urls[0] if urls else np.nan

work["image_url"] = work["image_urls_raw"].apply(extract_image)

# === 5) COMPUTE PRICE ANOMALY Z-SCORE ===
grp = work.groupby("category")["price"]
work["category_median_price"] = grp.transform("median")
work["category_std_price"] = grp.transform("std").replace(0, np.nan)
work["price_anomaly_z"] = (work["price"] - work["category_median_price"]) / work["category_std_price"]
work["price_anomaly_z"] = work["price_anomaly_z"].fillna(0.0)

# === 6) OPTIONAL TEXT FOR CLIP ===
work["text_for_clip"] = (
    work["title"].astype(str) + " [SEP] " +
    work["description"].astype(str).str.slice(0, 600)
)

# === 7) SELECT FINAL TRAINING COLUMNS ===
final = work[[
    "title", "description", "category",
    "price", "msrp", "price_anomaly_z",
    "image_url", "text_for_clip"
]].reset_index(drop=True)

final.to_csv("base_features_with_image.csv", index=False)
print("✅ Saved as base_features_with_image.csv. Rows:", len(final))
final.head()